# Bonus 04 — LiteLLM: One Gateway in Front of Every Model

**Optional | After Lab 5 | Colab CPU | `OPENAI_API_KEY` (optional `GROQ_API_KEY`)**

---

By now you have pointed the same client at OpenAI, at Groq, at your own FastAPI server (Lab 5) and maybe at vLLM (Bonus 03). Every time, the app code named the provider: a `base_url`, a key, a model id.

That gets painful fast. A price change means editing code, and so does a provider outage at 2am or a new model that is better for one kind of question. A **gateway** moves those decisions out of the app. The app asks for `"fast"` or `"quality"`; a routing table somewhere else decides what that means today, retries or falls back when something breaks, and keeps count of what it all costs.

**LiteLLM** is a widely used open-source gateway. It comes in two forms, and you will use both:

1. **A Python library**: one `completion()` function that speaks to many providers with OpenAI-shaped messages.
2. **A proxy server**: the same thing as a running service with its own OpenAI-compatible URL. It is Lab 5's idea with the routing built in.

## What you will walk out with

1. The same request sent through LiteLLM, and what a price per call looks like.
2. A routing table that the app refers to by name.
3. A provider that fails, and a fallback that answers anyway.
4. The gateway running as a server on this machine, called with the plain OpenAI client.

In [ ]:
import sys
%pip install -q uv
!uv pip install -q --python {sys.executable} "litellm[proxy]" openai pandas httpx python-dotenv

In [ ]:
import os
try:
    from google.colab import userdata          # Colab: read the Secret you added
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv             # local: read .env in the repo root
    load_dotenv()
assert os.environ.get("OPENAI_API_KEY"), "Add OPENAI_API_KEY as a Colab Secret or to .env"

OPENAI_API_KEY  = os.environ["OPENAI_API_KEY"]
OPENAI_BASE_URL = "https://api.openai.com/v1"
DEFAULT_MODEL   = "gpt-4o-mini"

from openai import OpenAI
client = OpenAI(api_key=OPENAI_API_KEY, base_url=OPENAI_BASE_URL)
print(f"Ready — {DEFAULT_MODEL} at {OPENAI_BASE_URL}")

Groq is optional. If a `GROQ_API_KEY` secret exists, LiteLLM reads it from the environment and a third route appears in section 2.

In [ ]:
try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
except Exception:                      # not on Colab, or no such secret
    GROQ_API_KEY = os.environ.get("GROQ_API_KEY")

if GROQ_API_KEY:
    os.environ["GROQ_API_KEY"] = GROQ_API_KEY
print("Groq route available:", bool(GROQ_API_KEY))

---

## 1. The same request, through LiteLLM

First the direct call you have made since Lab 1A, then the same call through `litellm.completion`. The only new thing is the model string: `openai/gpt-4o-mini`. The part before the slash is the provider; change it and the same `messages` go somewhere else (`groq/...`, `anthropic/...`, `hosted_vllm/...`).

In [ ]:
messages = [
    {"role": "system", "content": "You are a concise LLM deployment coach."},
    {"role": "user", "content": "When should a team add an LLM gateway? Three bullets."},
]
direct = client.chat.completions.create(model=DEFAULT_MODEL, messages=messages, temperature=0.2)
print(direct.choices[0].message.content)

In [ ]:
from litellm import completion, completion_cost

r = completion(model=f"openai/{DEFAULT_MODEL}", messages=messages, temperature=0.2)
print(r.choices[0].message.content)
print()
print(f"tokens: {r.usage.prompt_tokens} in, {r.usage.completion_tokens} out")
print(f"cost  : ${completion_cost(completion_response=r):.6f}")

**Checkpoint:** the same kind of answer, plus something the plain client never told you: a price. LiteLLM keeps a table of per-token prices for hundreds of models, so it can turn every response's token counts into dollars. Multiply that by your daily traffic and you have the number your finance team will ask about.

---

## 2. Routes, not model names

In the app, you want names that describe what you need, not who provides it. The table below maps those names to real models. In production it lives in a config file or a gateway UI, and changing it needs no deploy.

In [ ]:
import time
import pandas as pd

ROUTES = {
    "fast":    f"openai/{DEFAULT_MODEL}",
    "quality": "openai/gpt-4o",
}
if GROQ_API_KEY:
    ROUTES["groq"] = "groq/openai/gpt-oss-20b"

call_log = []

def gateway_chat(route, prompt):
    t0 = time.time()
    r = completion(model=ROUTES[route], temperature=0.2,
                   messages=[{"role": "system", "content": "You are a concise LLM deployment coach."},
                             {"role": "user", "content": prompt}])
    call_log.append({"route": route, "model": ROUTES[route], "ms": int((time.time() - t0) * 1000),
                     "tokens_in": r.usage.prompt_tokens, "tokens_out": r.usage.completion_tokens,
                     "cost_usd": completion_cost(completion_response=r)})
    return r.choices[0].message.content

In [ ]:
question = "Explain fallback routing in two sentences."
for route in ROUTES:
    print(f"[{route}]", gateway_chat(route, question), "\n")

pd.DataFrame(call_log)

**Checkpoint:** look at the `cost_usd` column. The same question costs well over ten times as much on `quality` as on `fast`. That ratio is why routing exists: send the easy majority of traffic to the cheap route and save the expensive one for questions that need it. Deciding *which* questions need it is your product logic; the gateway just makes the switch cheap.

---

## 3. When a provider fails

Providers go down, rate-limit you and time out. With a plain client, that is an exception in your app. With a gateway it is a line of config.

LiteLLM's `Router` holds several **deployments** and a **fallback** rule. To see it work, we break the primary on purpose: its API key is wrong. The backup is the same model with the real key. In real life the backup would be a different provider or region.

In [ ]:
from litellm import Router

router = Router(
    model_list=[
        {"model_name": "chat",        "model_info": {"id": "primary"},
         "litellm_params": {"model": f"openai/{DEFAULT_MODEL}", "api_key": "sk-wrong-on-purpose"}},
        {"model_name": "chat-backup", "model_info": {"id": "backup"},
         "litellm_params": {"model": f"openai/{DEFAULT_MODEL}"}},     # real key, from the environment
    ],
    fallbacks=[{"chat": ["chat-backup"]}],
    num_retries=0,                    # fail over at once; real setups retry a couple of times first
)

In [ ]:
r = router.completion(model="chat", messages=[{"role": "user", "content": "In one sentence: why do gateways have fallbacks?"}])

print(r.choices[0].message.content)
print("answered by:", r._hidden_params["model_id"])

**Checkpoint:** `answered by: backup`. The call to `chat` hit the broken primary, got an authentication error, and LiteLLM retried it on `chat-backup`, all inside one function call. You may see LiteLLM print the primary's error above the answer. That log line is the gateway telling you it did its job, and in production it is what you alert on.

---

## 4. Streaming passes through

Nothing about the gateway changes streaming: same chunks, same `delta.content`.

In [ ]:
from litellm import Router

router = Router(
    model_list=[
        {"model_name": "chat",        "model_info": {"id": "primary"},
         "litellm_params": {"model": f"openai/{DEFAULT_MODEL}", "api_key": "sk-wrong-on-purpose"}},
        {"model_name": "chat-backup", "model_info": {"id": "backup"},
         "litellm_params": {"model": f"openai/{DEFAULT_MODEL}"}},     # real key, from the environment
    ],
    fallbacks=[{"chat": ["chat-backup"]}],
    num_retries=0,                    # fail over at once; real setups retry a couple of times first
)

---

## 5. The gateway as a server

So far LiteLLM lived inside this notebook's Python. The more common setup in production is the **proxy**: LiteLLM running as its own service, with its own OpenAI-compatible URL, that every app in the company calls. Keys, routes, fallbacks and budgets live in one place, and apps only know one `base_url`.

It is Lab 5's server with the routing built in, and it starts the same way: a config file, a background process, a health check.

In [ ]:
%%writefile litellm_config.yaml
model_list:
  - model_name: fast
    litellm_params:
      model: openai/gpt-4o-mini
      api_key: os.environ/OPENAI_API_KEY
  - model_name: quality
    litellm_params:
      model: openai/gpt-4o
      api_key: os.environ/OPENAI_API_KEY
  # A Bonus 03 vLLM server would be one more entry:
  # - model_name: our-gpu
  #   litellm_params:
  #     model: hosted_vllm/qwen-lab4
  #     api_base: http://127.0.0.1:8000/v1

`os.environ/OPENAI_API_KEY` tells LiteLLM to read the key from its environment when it starts. The key is never written into the config, the same rule as Lab 10's Docker image.

In [ ]:
import os, shutil, subprocess, sys, httpx

LITELLM = shutil.which("litellm") or os.path.join(os.path.dirname(sys.executable), "litellm")
gateway = subprocess.Popen([LITELLM, "--config", "litellm_config.yaml", "--port", "4000"],
                           stdout=open("litellm.log", "w"), stderr=subprocess.STDOUT)

for _ in range(60):                                     # up to 2 minutes
    try:
        if httpx.get("http://127.0.0.1:4000/health/liveliness", timeout=2).status_code == 200:
            print("gateway is up on port 4000")
            break
    except httpx.HTTPError:
        pass
    time.sleep(2)

Now call it the way any app in your company would: the ordinary OpenAI client, one `base_url`, and a route name where the model id used to be.

In [ ]:
from openai import OpenAI

gw = OpenAI(base_url="http://127.0.0.1:4000/v1", api_key="anything")   # no gateway key set in this demo
print("routes:", [m.id for m in gw.models.list().data])

for route in ["fast", "quality"]:
    r = gw.chat.completions.create(model=route, messages=[{"role": "user", "content": "In five words: what is an LLM gateway?"}])
    print(f"{route:>8} → response says model={r.model!r}: {r.choices[0].message.content}")

**Checkpoint:** the client asked for `fast`, and the response says `fast` too. Nowhere on the client side does `gpt-4o-mini` appear. Which model and which provider answered is now entirely the gateway's business, and changing it means editing `litellm_config.yaml` and restarting one process, not redeploying every app.

A production proxy adds what this demo leaves out: a master key and per-team virtual keys, spend limits, a database of every call, and a dashboard. All of it is configuration on the same process.

In [ ]:
gateway.terminate()
gateway.wait()
print("gateway stopped")

---

## Bonus 04 complete

- [ ] A call through `litellm.completion`, with its price
- [ ] A route table, and the cost ratio between routes
- [ ] A broken primary and a fallback that answered
- [ ] The LiteLLM proxy running on this machine, called with the plain OpenAI client

## Stretch

1. **Route by question.** Write `pick_route(prompt)` that sends prompts over 50 words to `quality` and everything else to `fast`. Run ten mixed prompts and compare the bill with sending everything to `quality`.
2. **Retries before fallbacks.** Set `num_retries=2` on the Router and time the broken call. What did the retries cost you in latency?
3. **Three providers.** Add a Groq deployment to `litellm_config.yaml` (`groq/openai/gpt-oss-20b`, `api_key: os.environ/GROQ_API_KEY`), restart the gateway and call it by its new route name.
4. **Your own GPU behind the gateway.** With Bonus 03's vLLM server running, uncomment the `our-gpu` entry and call `model="our-gpu"`.

Next: [Bonus 05 — Ollama](05_ollama_local.md), a local backend on your laptop behind the same client.